# Scraping

This includes all steps taken and tests done to extract data from property24

## Suburb Lookup file

This extracts all suburbs and cities that are categorised in property24. For this project, I will be look at gauteng only.

In [1]:
import pandas as pd
import numpy as np
import requests
import re
import math
from pathlib import Path
from bs4 import BeautifulSoup
import json
import time
import random
import os
from datetime import datetime

# display settings
pd.set_option('display.max_columns', None)

In [2]:
base_url = "https://www.property24.com"
province_name = "gauteng"
province_id = 1

In [3]:
import logging
from fake_useragent import UserAgent

# 1. Initialize global tools outside the function
# This keeps the "identity" consistent during the script run
ua = UserAgent(browsers=['chrome', 'edge', 'firefox'], os=['windows'])
session = requests.Session()

# Create a local list to avoid the "Error occurred" warning
SAFE_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/121.0"
]

def fetch_data(url):
    """Fetches HTML using a persistent session and rotating desktop headers."""
    
    # Check if we are still using a local session
    global session

    # 2. Build Stealth Headers
    # We rotate the User-Agent but keep it restricted to Desktop to avoid layout breaks
    headers = {
        "User-Agent": random.choice(SAFE_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Referer": "https://www.property24.com/",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1"
    }

    try:
        # 3. Randomized Delay
        # Increased range slightly to be more "human"
        wait_time = random.uniform(13.5, 28.2)
        logging.info(f"Waiting {wait_time:.2f}s before fetching...")
        time.sleep(wait_time)

        # 4. Perform the Request
        response = session.get(url, headers=headers, timeout=15)

        # 5. Specific Error Handling for Blocks
        if response.status_code == 503:
            logging.error(f"❌ 503 Service Unavailable: Property24 is rate-limiting you at {url}")
            return None
        if response.status_code == 403:
            logging.error(f"❌ 403 Forbidden: Your IP might be temporarily flagged.")
            return None
        
        # Raise other errors (404, 500, etc.)
        response.raise_for_status()
        
        return response.text

    except requests.exceptions.HTTPError as e:
        logging.error(f"HTTP Error: {e}")
    except requests.exceptions.ConnectionError:
        logging.error("Connection Error: Check your internet or if the server is down.")
    except Exception as e:
        logging.error(f"Unexpected error fetching {url}: {e}")
    
    return None

In [4]:
# def fetch_data(url):
#     # 1. Generate a random User-Agent for this specific request
#     headers = {'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}
    
#     # 2. Add a random sleep timer (the "Human Pause")
#     # A good range for SA sites is 3 to 7 seconds. 
#     # Anything under 2 seconds is risky for large batches.
#     time.sleep(random.uniform(3, 7)) 
    
#     response = requests.get(url, headers=headers)
#     return response.text if response.status_code == 200 else None

In [5]:
def extract_suburbs():
    cities = []

    url = f"{base_url}/to-rent/all-cities/{province_name}/1"
    html = fetch_data(url)
    soup = BeautifulSoup(html, "html.parser")

    labels = soup.find_all('label', class_='checkbox')
    for label in labels:
        checkbox = label.find("input", {"name": "AllCityIds"})
        link_tag = label.find("a")

        if checkbox and link_tag:
            city_id = checkbox.get("value")
            name = link_tag.text.strip().lower()

            link = f"{base_url}/to-rent/all-suburbs/{name}/{province_name}/{city_id}"

            sec_html = fetch_data(link)
            sec_soup = BeautifulSoup(sec_html, "html.parser")

            suburbs = sec_soup.find_all('label', class_='checkbox')
            for suburb in suburbs:
                sec_checkbox = suburb.find("input", {"name": "AllSuburbIds"})
                sec_link_tag = suburb.find("a")

                if sec_checkbox and sec_link_tag:
                    suburb_id = sec_checkbox.get("value")
                    suburb_name = sec_link_tag.text.strip().lower()

                    cities.append({
                        "suburb_id":suburb_id,
                        "suburb_name":suburb_name,
                        "city_id":city_id,
                        "city_name":name,
                        "province_id":province_id,
                        "province_name":province_name
                    })
    
    return cities

In [6]:
CITY_TO_MUNICIPALITY = {
    # City of Tshwane
    "pretoria": "City of Tshwane",
    "centurion": "City of Tshwane",
    "akasia": "City of Tshwane",
    "ga-rankuwa": "City of Tshwane",
    "soshanguve": "City of Tshwane",
    "mabopane": "City of Tshwane",
    "hammanskraal": "City of Tshwane",
    "temba": "City of Tshwane",
    "cullinan": "City of Tshwane",
    "bronkhorstspruit": "City of Tshwane",
    "wonderboom": "City of Tshwane",

    # City of Johannesburg
    "johannesburg": "City of Johannesburg",
    "sandton": "City of Johannesburg",
    "randburg": "City of Johannesburg",
    "roodepoort": "City of Johannesburg",
    "midrand": "City of Johannesburg",
    "soweto": "City of Johannesburg",
    "bedfordview": "City of Johannesburg",
    "eikenhof": "City of Johannesburg",
    "walkerville": "City of Johannesburg",

    # Ekurhuleni
    "alberton": "Ekurhuleni",
    "benoni": "Ekurhuleni",
    "boksburg": "Ekurhuleni",
    "brakpan": "Ekurhuleni",
    "kempton park": "Ekurhuleni",
    "germiston": "Ekurhuleni",
    "springs": "Ekurhuleni",
    "katlehong": "Ekurhuleni",
    "tembisa": "Ekurhuleni",
    "edenvale": "Ekurhuleni",
    "heidelberg": "Ekurhuleni",
    "nigel": "Ekurhuleni",

    # Sedibeng
    "vereeniging": "Sedibeng",
    "vanderbijlpark": "Sedibeng",
    "meyerton": "Sedibeng",
    "sebokeng": "Sedibeng",
    "evaton": "Sedibeng",
    "vaal marina": "Sedibeng",

    # West Rand
    "krugersdorp": "West Rand",
    "randfontein": "West Rand",
    "carletonville": "West Rand",
    "westonaria": "West Rand",
    "fochville": "West Rand",
}

In [7]:
# file_path = Path("../data/processed/suburb_lookup.csv")

# if file_path.exists():
#     print("Found it!")
#     cities_df = pd.read_csv("../data/processed/suburb_lookup.csv")
# else:
#     print("File is missing.")
#     cities_df = pd.DataFrame(extract_suburbs())
#     cities_df["municipality"] = (
#         cities_df["city_name"]
#         .str.strip()
#         .str.lower()
#         .map(CITY_TO_MUNICIPALITY)
#         .fillna("Unknown")
#     )
#     cities_df.to_csv("../data/processed/suburb_lookup.csv", index=False)


In [8]:
# cities_df

In [9]:
df = pd.read_parquet("../data/0_metadata/suburb_lookup.parquet")

In [10]:
df

,suburb_id,suburb_name,city_id,city_name,province_id,province_name,municipality,last_scraped
0,3990,amandasig,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
1,4002,chantelle,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
2,4012,clarina,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
3,4102,doreg ah,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
4,429,eldorette,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
...,...,...,...,...,...,...,...,...
2068,346,rooiwal ah,2473,wonderboom,1,gauteng,City of Tshwane,2026-02-06
2069,347,vasfontein ah,2473,wonderboom,1,gauteng,City of Tshwane,2026-02-06
2070,17,walmansthal ah,2473,wonderboom,1,gauteng,City of Tshwane,2026-02-06
2071,55,waterval ah,2473,wonderboom,1,gauteng,City of Tshwane,None


In [11]:
df[df['last_scraped'].isnull()]

,suburb_id,suburb_name,city_id,city_name,province_id,province_name,municipality,last_scraped
2071,55,waterval ah,2473,wonderboom,1,gauteng,City of Tshwane,None
2072,51,welgevonden,2473,wonderboom,1,gauteng,City of Tshwane,None


In [12]:
df[df['city_name'] == "akasia"]

,suburb_id,suburb_name,city_id,city_name,province_id,province_name,municipality,last_scraped
0,3990,amandasig,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
1,4002,chantelle,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
2,4012,clarina,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
3,4102,doreg ah,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
4,429,eldorette,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
5,10500,hartebeeshoek,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
6,10635,heatherdale ah,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
7,11007,heatherview,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
8,334,hesteapark,2462,akasia,1,gauteng,City of Tshwane,2026-02-02
9,335,karenpark,2462,akasia,1,gauteng,City of Tshwane,2026-02-02


## Rentals extraction

The rentals will be extracted from property24 in batches by suburb, then saved to a json file.

In [13]:
def save_property_snapshot(df, suburb_name):
    # 1. Create a timestamp for the folder structure
    now = datetime.now()
    year = now.strftime("%Y")
    month = now.strftime("%m")
    
    # 2. Define the directory path (e.g., data/2026/01/)
    directory = f"data/year={year}/month={month}"
    
    # Create the folder if it doesn't exist
    if not os.path.exists(directory):
        os.makedirs(directory)
    
    # 3. Define the filename based on the suburb
    file_path = f"{directory}/{suburb_name.lower().replace(' ', '_')}.parquet"
    
    # 4. Save to Parquet
    # 'index=False' prevents an extra column of row numbers being saved
    df.to_parquet(file_path, engine='pyarrow', compression='snappy', index=False)
    
    print(f"✅ Successfully saved {len(df)} listings to {file_path}")

# Example Usage:
# df_sandton = scrape_property24("Sandton")
# save_property_snapshot(df_sandton, "Sandton")

Test

In [14]:
# setup variables
data = []

city_name = "pretoria"
suburb_id = 33189
suburb_name = "silver lakes"

suburb_name_norm = suburb_name.replace(' ', '-')
city_name_norm = city_name.replace(' ', '-')

In [15]:
def get_points_of_interest(listing_number):
    """Fetches the hidden POI data via direct AJAX link."""
    poi_url = f"https://www.property24.com/ListingReadOnly/PointsOfInterestForListing?ListingNumber={listing_number}"
    try:
        # It's better to pass a session here if you can
        response_text = fetch_data(poi_url) 
        if response_text:
            # We don't even need to soup it yet, just save the raw HTML string
            return response_text 
    except Exception as e:
        # logger.error(f"Failed to fetch POI for {listing_number}: {e}")
        print(f"Failed to fetch POI for {listing_number}: {e}")
    return None

In [16]:
def get_listings_data(listing_cards, base_url):
    """Deep scrapes individual property pages."""
    results = []
    for listing_card in listing_cards:
        listing_number = listing_card.get('data-listing-number')
        link_tag = listing_card.find("a")

        if not listing_number or not link_tag:
            print(f" OOPs oh nooo! {listing_number}")
            continue

        link_url = link_tag.get('href')
        listing_url = base_url + link_url
        
        # logger.info(f"--- 🏠 Deep scraping property {listing_number}")
        print(f"--- 🏠 Deep scraping property {listing_number}")

        # 🚨 THE RECOVERY LIFT: 
        # Increase delay significantly. 10-20 seconds between HOUSES.
        time.sleep(random.uniform(10.0, 20.0))
        
        # 1. Fetch Property Page
        listing_html = fetch_data(listing_url)
        
        # 2. Fetch POI
        # Small delay to avoid looking like a bot
        time.sleep(random.uniform(0.5, 1.5)) 
        # listing_poi = get_points_of_interest(listing_number)

        results.append({
            "listing_number": listing_number,
            "link": link_url,
            "page_data": listing_html, # Save raw HTML for Bronze
            # "poi_data": listing_poi,
            "extracted_at": datetime.now().isoformat()
        })
    return results           

In [17]:
def get_suburb_listings(suburb_id, suburb_name, city_name, province_name="gauteng", listing_type="to-rent"):
    """
    Main entry point. 
    listing_type can be 'to-rent' or 'for-sale'
    """
    base_url = "https://www.property24.com"
    all_listings = []
    
    sub_norm = suburb_name.lower().replace(' ', '-')
    city_norm = city_name.lower().replace(' ', '-')
    
    # Dynamic URL for Sale or Rent
    url = f"{base_url}/{listing_type}/{sub_norm}/{city_norm}/{province_name}/{suburb_id}"
    
    # logger.info(f"🚀 Starting scrape for {suburb_name} ({listing_type})")
    print(f"🚀 Starting scrape for {suburb_name} ({listing_type})")
    
    html = fetch_data(url)
    if not html:
        # logger.error(f"Could not reach {url}")
        print(f"Could not reach {url}")
        return None

    soup = BeautifulSoup(html, "html.parser")
    
    # Pagination Logic
    try:
        pager = soup.find('div', class_='p24_topPager')
        if pager:
            bold_spans = pager.find_all('span', class_='p24_bold')
            tot_num = int(bold_spans[-1].text.replace(' ', '')) # Handle space in "1 200"
            num_pages = math.ceil(tot_num / 20)
        else:
            num_pages = 1
    except Exception as e:
        # logger.warning(f"Could not determine page count, defaulting to 1. Error: {e}")
        print(f"Could not determine page count, defaulting to 1. Error: {e}")
        num_pages = 1

    # Loop Pages
    for i in range(1, num_pages + 1):
        page_url = f"{url}/p{i}"
        # logger.info(f"📄 Scraping Page {i} of {num_pages}")
        print(f"📄 Scraping Page {i} of {num_pages}")
        
        page_html = fetch_data(page_url)
        if page_html:
            p_soup = BeautifulSoup(page_html, "html.parser")
            listing_cards = p_soup.find_all('div', class_='p24_tileContainer')
            
            # Use EXTEND to keep a flat list of dictionaries
            all_listings.extend(get_listings_data(listing_cards, base_url))
            
        # Respectful delay between search pages
        time.sleep(random.uniform(2, 4))

    return {
        "suburb_id": suburb_id,
        "suburb_name": suburb_name,
        "city_name": city_name,
        "listing_type": listing_type,
        "scraped_at": datetime.now().isoformat(),
        "listings_count": len(all_listings),
        "listings": all_listings
    }

In [18]:
def get_suburb_listings(suburb_id, suburb_name, city_name, province_name="gauteng", listing_type="to-rent"):
    """
    Main entry point. 
    listing_type can be 'to-rent' or 'for-sale'
    """
    base_url = "https://www.property24.com"
    all_listings = []
    
    sub_norm = suburb_name.lower().replace(' ', '-')
    city_norm = city_name.lower().replace(' ', '-')
    
    # Dynamic URL for Sale or Rent
    url = f"{base_url}/{listing_type}/{sub_norm}/{city_norm}/{province_name}/{suburb_id}"
    
    # logger.info(f"🚀 Starting scrape for {suburb_name} ({listing_type})")
    print(f"🚀 Starting scrape for {suburb_name} ({listing_type})")
    
    html = fetch_data(url)
    if not html:
        # logger.error(f"Could not reach {url}")
        print(f"Could not reach {url}")
        return None

    soup = BeautifulSoup(html, "html.parser")
    
    # Pagination Logic
    try:
        pager = soup.find('div', class_='p24_topPager')
        if pager:
            bold_spans = pager.find_all('span', class_='p24_bold')
            tot_num = int(bold_spans[-1].text.replace(' ', '')) # Handle space in "1 200"
            num_pages = math.ceil(tot_num / 20)
        else:
            num_pages = 1
    except Exception as e:
        # logger.warning(f"Could not determine page count, defaulting to 1. Error: {e}")
        print(f"Could not determine page count, defaulting to 1. Error: {e}")
        num_pages = 1

    # Loop Pages
    for i in range(1, num_pages + 1):
        page_url = f"{url}/p{i}"
        
        # logger.info(f"📄 Scraping Page {i} of {num_pages}")
        print(f"📄 Scraping Page {i} of {num_pages}")
        
        page_html = fetch_data(page_url)
        if page_html:
            p_soup = BeautifulSoup(page_html, "html.parser")
            listing_cards = p_soup.find_all('div', class_='p24_tileContainer')
            
            # Use EXTEND to keep a flat list of dictionaries
            # all_listings.extend(get_listings_data(listing_cards, base_url))
            all_listings.extend(listing_cards)
            
        # Respectful delay between search pages
        time.sleep(random.uniform(2, 4))

    return {
        "suburb_id": suburb_id,
        "suburb_name": suburb_name,
        "city_name": city_name,
        "listing_type": listing_type,
        "scraped_at": datetime.now().isoformat(),
        "listings_count": len(all_listings),
        "listings": all_listings
    }

In [19]:
# get_suburb_listings(suburb_id=10834, suburb_name="mooikloof ridge", city_name="pretoria")

In [20]:
import asyncio
import random
import math
from datetime import datetime

from playwright.async_api import async_playwright
from playwright_stealth import Stealth
from bs4 import BeautifulSoup


# -----------------------------
# Constants for "Human" behavior
# -----------------------------
SAFE_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
]


# -----------------------------
# Stealth instance (FIXED)
# -----------------------------
stealth = Stealth()


# -----------------------------
# POI fetch (FIXED)
# -----------------------------
async def fetch_poi_data(page, listing_number):
    """
    Fetch POI data using Playwright's request API.
    Uses the browser session cookies.
    """
    poi_url = (
        "https://www.property24.com/"
        f"ListingReadOnly/PointsOfInterestForListing?ListingNumber={listing_number}"
    )

    try:
        response = await page.context.request.get(poi_url)
        return await response.text()
    except Exception as e:
        print(f"Failed POI fetch for {listing_number}: {e}")
        return None


# -----------------------------
# Deep scrape listings (FIXED)
# -----------------------------
async def deep_scrape_listings(context, listing_links, base_url):
    """
    Uses a NEW PAGE per listing to reduce fingerprinting.
    """
    results = []

    for link in listing_links:
        page = await context.new_page()
        await stealth.apply(page)

        full_url = base_url + link["href"]
        listing_num = link["num"]

        print(f"--- 🏠 Deep scraping property {listing_num}")

        await asyncio.sleep(random.uniform(5.5, 12.0))

        try:
            await page.goto(
                full_url,
                wait_until="domcontentloaded",
                timeout=60_000,
            )

            listing_html = await page.content()
            poi_html = await fetch_poi_data(page, listing_num)

            results.append(
                {
                    "listing_number": listing_num,
                    "url": full_url,
                    "page_data": listing_html,
                    "poi_data": poi_html,
                    "scraped_at": datetime.now().isoformat(),
                }
            )

        except Exception as e:
            print(f"Error scraping {full_url}: {e}")

        finally:
            await page.close()

    return results


# -----------------------------
# Main scraper
# -----------------------------
async def run_property_scraper(
    suburb_id,
    suburb_name,
    city_name,
    province="gauteng",
    listing_type="to-rent",
):
    base_url = "https://www.property24.com"

    sub_norm = suburb_name.lower().replace(" ", "-")
    city_norm = city_name.lower().replace(" ", "-")

    start_url = (
        f"{base_url}/{listing_type}/"
        f"{sub_norm}/{city_norm}/{province}/{suburb_id}"
    )

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)

        context = await browser.new_context(
            user_agent=random.choice(SAFE_AGENTS),
            viewport={
                "width": random.randint(1280, 1920),
                "height": random.randint(720, 1080),
            },
            locale="en-ZA",
            timezone_id="Africa/Johannesburg",
        )

        page = await context.new_page()
        await stealth.apply(page)

        print(f"🚀 Starting scrape: {start_url}")
        await page.goto(start_url, wait_until="networkidle")

        content = await page.content()
        soup = BeautifulSoup(content, "html.parser")

        # -----------------------------
        # Pagination (GUARDED)
        # -----------------------------
        try:
            pager = soup.find("div", class_="p24_topPager")
            spans = pager.find_all("span", class_="p24_bold") if pager else []
            tot_num = int(spans[-1].text.replace(" ", "")) if spans else 0
            num_pages = math.ceil(tot_num / 20) if tot_num else 1
        except Exception:
            num_pages = 1

        all_collected_data = []

        for i in range(1, num_pages + 1):
            page_url = f"{start_url}/p{i}"
            print(f"📄 Processing Search Page {i} of {num_pages}")

            if i > 1:
                await page.goto(page_url, wait_until="domcontentloaded")
                content = await page.content()
                soup = BeautifulSoup(content, "html.parser")

            listing_cards = soup.find_all("div", class_="p24_tileContainer")
            links_to_scrape = []

            for card in listing_cards:
                l_num = card.get("data-listing-number")
                a_tag = card.find("a")
                l_link = a_tag.get("href") if a_tag else None

                if l_num and l_link:
                    links_to_scrape.append(
                        {
                            "num": l_num,
                            "href": l_link,
                        }
                    )

            page_results = await deep_scrape_listings(
                context,
                links_to_scrape,
                base_url,
            )
            all_collected_data.extend(page_results)

            await asyncio.sleep(random.uniform(2, 5))

        await browser.close()
        return all_collected_data


# -----------------------------
# Example run
# -----------------------------
# if __name__ == "__main__":
#     results = asyncio.run(
#         run_property_scraper(
#             suburb_id="1234",
#             suburb_name="Sandton",
#             city_name="Johannesburg",
#         )
#     )


In [21]:
# results = asyncio.run(run_property_scraper(suburb_id=10834, suburb_name="mooikloof ridge", city_name="pretoria"))

In [22]:
{
  "id": "111446108",
  "listingNumber": "111446108",
  "price": "R 39 000 000", # p4_price
  "url": "https://www.property24.com/for-sale/val-de-vie-estate/paarl/western-cape/11726/111446108",
  # p24_listingAbout under h5
  "title": "5 Bedroom House for Sale in Val De Vie Estate",
  # p24_expandedText
  "description": "Double Storey",
  # class_=p24_listingCard
  # p24_keyFeaturesContainer 1
  "bedrooms": "5",
  "bathrooms": "5",
  "parkings": "4",
  "garages": "2",
  "parking": "2",
  "typeOfProperty": "House",
  "listingDate": "12 June 2025",
  "erfSize": "1 714 m²",
  "floorSize": "870 m²",
  "petsAllowed": "Yes",
  "zoning": "General Residential",
  "rooms": {
    "bedrooms": "5",
    "bathrooms": "5, Guest Toilet, En suite",
    "kitchen": "1",
    "office": "1",
    "diningRoom": "1",
    "lounge": "1"
  },
  # p24_keyFeaturesContainer 1
  "features": [
    "Air Conditioning",
    "Alarm System",
    "Balcony",
    "Braai Room",
    "Built In Cupboards"
  ],
  "pointOfInterest": [
    {
      "category": "Schools",
      "name": "Val de Vie Primary School",
      "distance": "2.5 km"
    },
    {
      "category": "Shopping",
      "name": "Boland Centre",
      "distance": "15 km"
    }
  ]
}

{'id': '111446108',
 'listingNumber': '111446108',
 'price': 'R 39 000 000',
 'url': 'https://www.property24.com/for-sale/val-de-vie-estate/paarl/western-cape/11726/111446108',
 'title': '5 Bedroom House for Sale in Val De Vie Estate',
 'description': 'Double Storey',
 'bedrooms': '5',
 'bathrooms': '5',
 'parkings': '4',
 'garages': '2',
 'parking': '2',
 'typeOfProperty': 'House',
 'listingDate': '12 June 2025',
 'erfSize': '1 714 m²',
 'floorSize': '870 m²',
 'petsAllowed': 'Yes',
 'zoning': 'General Residential',
 'rooms': {'bedrooms': '5',
  'bathrooms': '5, Guest Toilet, En suite',
  'kitchen': '1',
  'office': '1',
  'diningRoom': '1',
  'lounge': '1'},
 'features': ['Air Conditioning',
  'Alarm System',
  'Balcony',
  'Braai Room',
  'Built In Cupboards'],
 'pointOfInterest': [{'category': 'Schools',
   'name': 'Val de Vie Primary School',
   'distance': '2.5 km'},
  {'category': 'Shopping', 'name': 'Boland Centre', 'distance': '15 km'}]}

In [23]:
{
    "descriptionHeader": "The Crowning Jewel of the Esteemed Val de Vie Gentlemen’s Estate",
    "description": "Distinguished Country Manor Set Amidst Unsurpassed Splendour.\n\nPresenting a rare confluence of refined living and verdant charm, this magnificent estate—encompassing an unsurpassed expanse of 25,635 square metres—lies gracefully within the noble confines of the Val de Vie Gentlemen’s Estate, a sanctuary of prestige and quiet dignity.\n\nHerein lies a residence of exceptional stature, wherein the comforts of modernity are seamlessly woven into the very fabric of nature. The property boasts a multitude of well-appointed bedroom suites, formal drawing rooms with both indoor and alfresco spaces elegantly customised for the entertainment of the most discerning residents. \n\nA lap pool sheltered by vibrant lilac trellises of jacaranda blossoms, accompanied by a piping hot jacuzzi, commands a splendid vista of the noble Simonsberg mountain range, offering an ever-changing private vista of nature’s majesty at your doorstep.\n\nEnsconced within a privately governed estate, the property assures peace of mind and security of the highest order, with vigilant patrols by trained guards, precision controlled access points, and ever-watchful world class surveillance software, ensuring that your Family remains sheltered and at ease.\n\nThe Val de Vie Estate stands as a bastion of protection and privilege, bestowing upon its residents both safety and the serenity of a pastoral lifestyle.\n\nOne is invited to partake in the finest pursuits—unwind with colleagues upon a championship golf course, or enjoy a tranquil repose beside the shimmering waters of the private pool... while every equestrian endeavour is catered for by the world class stables and magnificent polo facilities. \n\nThe principal residence spans over 2,000 square metres, comprising multiple barbecue enclaves, a dedicated cinema theatre and seamless multiple outdoor living quarters. The entertainment wing flows with grace into a landscaped garden adorned by an eco-conscious lake, a central boma with fire pit, a wood-fired jacuzzi, an open-air cinema, and a heated swimming pool, all harmoniously arranged to foster both festivity and repose.\n\nThe master suite is a haven unto itself, featuring an opulent spa bath and a dressing room of elegant proportions. The children’s wing houses four en-suite bedrooms, affording comfort and privacy to the younger members of your household.\n\nAdditional accommodation includes a soundproofed cinema or music parlour, a self-contained apartment, a home office, two guest suites, and a staff quarters comprising four en-suite rooms and a dedicated living area.\n\nLet not this rare offering elude you—a private Legacy Property: where the grandeur of nature meets the hallmarks of luxury. \nHere, in this cherished haven, one may lead a life of both distinction and discretion.",
    "keyFeatures": [
        {
            "text": "Bedrooms",
            "icon": "/Content/images/Optimized/Icons/icon_bed_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30",
            "value": "12"
        },
        {
            "text": "Bathrooms",
            "icon": "/Content/images/Optimized/Icons/icon_bathroom_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30",
            "value": "12.5"
        },
        {
            "text": "Garages",
            "icon": "/Content/images/Optimized/Icons/icon_garage_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30",
            "value": "9"
        },
        {
            "text": "Parking",
            "icon": "/Content/images/Optimized/Icons/icon_secure_parking_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30",
            "value": "10"
        },
        {
            "text": "Study",
            "icon": "/Content/images/Optimized/Icons/icon_study_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        },
        {
            "text": "Pet Friendly",
            "icon": "/Content/images/Optimized/Icons/icon_pet_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        },
        {
            "text": "Flatlet",
            "icon": "/Content/images/Optimized/Icons/icon_flatlet_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        },
        {
            "text": "Garden",
            "icon": "/Content/images/Optimized/Icons/icon_garden_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        },
        {
            "text": "Borehole, Water Tank",
            "icon": "/Content/images/Optimized/Icons/icon_faucet_drip.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        },
        {
            "text": "Solar Panels, Solar Geyser",
            "icon": "/Content/images/Optimized/Icons/icon_solar_panel.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        },
        {
            "text": "Backup Battery / Inverter",
            "icon": "/Content/images/Optimized/Icons/icon_battery_bolt.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        }
    ],
    "keyFeaturesLeft": [
        {
            "text": "Bedrooms",
            "icon": "/Content/images/Optimized/Icons/icon_bed_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30",
            "value": "12"
        },
        {
            "text": "Bathrooms",
            "icon": "/Content/images/Optimized/Icons/icon_bathroom_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30",
            "value": "12.5"
        },
        {
            "text": "Garages",
            "icon": "/Content/images/Optimized/Icons/icon_garage_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30",
            "value": "9"
        },
        {
            "text": "Parking",
            "icon": "/Content/images/Optimized/Icons/icon_secure_parking_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30",
            "value": "10"
        },
        {
            "text": "Study",
            "icon": "/Content/images/Optimized/Icons/icon_study_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        },
        {
            "text": "Pet Friendly",
            "icon": "/Content/images/Optimized/Icons/icon_pet_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        }
    ],
    "keyFeaturesRight": [
        {
            "text": "Flatlet",
            "icon": "/Content/images/Optimized/Icons/icon_flatlet_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        },
        {
            "text": "Garden",
            "icon": "/Content/images/Optimized/Icons/icon_garden_blue.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        },
        {
            "text": "Borehole, Water Tank",
            "icon": "/Content/images/Optimized/Icons/icon_faucet_drip.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        },
        {
            "text": "Solar Panels, Solar Geyser",
            "icon": "/Content/images/Optimized/Icons/icon_solar_panel.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        },
        {
            "text": "Backup Battery / Inverter",
            "icon": "/Content/images/Optimized/Icons/icon_battery_bolt.svg?35-30-2E-33-38-2E-37-34-35-2E-30"
        }
    ],
    "details": [
        {
            "categoryItems": [
                {
                    "name": "Listing Number",
                    "values": [
                        "116173830"
                    ]
                },
                {
                    "name": "Type of Property",
                    "values": [
                        "House"
                    ]
                },
                {
                    "name": "List Date",
                    "values": [
                        "07 July 2025"
                    ]
                },
                {
                    "name": "Erf Size",
                    "values": [
                        "25 635 m²"
                    ]
                },
                {
                    "name": "Floor Size",
                    "values": [
                        "2335 m²"
                    ]
                },
                {
                    "name": "Pets Allowed",
                    "values": [
                        "Yes"
                    ]
                }
            ],
            "name": "Property Details"
        },
        {
            "categoryItems": [
                {
                    "name": "Bedroom",
                    "values": [
                        "12"
                    ]
                },
                {
                    "name": "Bathrooms",
                    "values": [
                        "12.5"
                    ]
                },
                {
                    "name": "Kitchens",
                    "values": [
                        "3"
                    ]
                },
                {
                    "name": "Office/study",
                    "values": [
                        "1"
                    ]
                },
                {
                    "name": "Domestic",
                    "values": [
                        "1"
                    ]
                },
                {
                    "name": "Reception Rooms",
                    "values": [
                        "8"
                    ]
                }
            ],
            "name": "Rooms"
        },
        {
            "categoryItems": [
                {
                    "name": "Garage",
                    "values": [
                        "9"
                    ]
                },
                {
                    "name": "Parking",
                    "values": [
                        "10"
                    ]
                },
                {
                    "name": "Garden",
                    "values": [
                        "Yes"
                    ]
                }
            ],
            "name": "External Features"
        },
        {
            "categoryItems": [
                {
                    "name": "Backup Power",
                    "values": [
                        "Backup Battery / Inverter"
                    ]
                }
            ],
            "name": "Building"
        },
        {
            "categoryItems": [
                {
                    "name": "Flatlet",
                    "values": [
                        "Yes"
                    ]
                }
            ],
            "name": "Other Features"
        }
    ],
    "pointsOfInterest": [
        {
            "id": 6,
            "name": "Transport and Public Services",
            "pointsOfInterestItems": [
                {
                    "distance": "1.85km",
                    "name": "Cillie"
                },
                {
                    "distance": "3.39km",
                    "name": "Simondium"
                },
                {
                    "distance": "4.69km",
                    "name": "Paarl"
                },
                {
                    "distance": "5.72km",
                    "name": "Fairview"
                },
                {
                    "distance": "6.37km",
                    "name": "Groot-Drakenstein"
                }
            ]
        },
        {
            "id": 1,
            "name": "Education",
            "pointsOfInterestItems": [
                {
                    "distance": "3.75km",
                    "name": "Simond Privaatskool"
                },
                {
                    "distance": "3.84km",
                    "name": "Courtrai Primary"
                },
                {
                    "distance": "4.00km",
                    "name": "Simondium Primary"
                },
                {
                    "distance": "4.83km",
                    "name": "Montessori @ Home Independent School"
                },
                ...
            ]
        },
        {
            "id": 5,
            "name": "Sports and Leisure",
            "pointsOfInterestItems": [
                {
                    "distance": "4.38km",
                    "name": "Boschenmeer Golf Course"
                },
                {
                    "distance": "7.01km",
                    "name": "Haweqwa Nature Reserve"
                },
                {
                    "distance": "8.27km",
                    "name": "Picnic spot"
                }
            ]
        },
        {
            "id": 2,
            "name": "Food and Entertainment",
            "pointsOfInterestItems": [
                {
                    "distance": "5.71km",
                    "name": "Fairview"
                },
                {
                    "distance": "7.07km",
                    "name": "Hennie's Paarl Restaurant"
                },
                {
                    "distance": "7.29km",
                    "name": "Cattle Baron - Paarl"
                }
            ]
        }
    ],
    "nextShowDate": "0001-01-01T00:00:00+02:00",
    "nextShowEndDate": "0001-01-01T00:00:00+02:00",
    "showDays": [],
    "listingUrl": "https://www.property24.com/for-sale/val-de-vie-estate/paarl/western-cape/11726/116173830",
    "sourceReference": "RL24794",
    "listingMedia": {
        "youTubeVideoTourId": "Lajz4VWPEQo",
        "matterportSpaceId": "",
        "eyeSpy360Id": ""
    },
    "auctionVenue": "",
    "hasVenue": false,
    "soldPrices": [
        {
            "year": 2025,
            "month": 7,
            "price": 22500000,
            "url": "https://www.property24.com/property-values/552-arena-place/val-de-vie-estate/paarl/western-cape/uuicbb7dkjccoayasjiyd2kr7dbhe3x7dskum76u5ljo2zf7coe4eyqc2hmy6mnxao2t4ilyy5uhc",
            "streetNumber": "552",
            "streetName": "Arena Place",
            "suburb": "Val de Vie Estate"
        },
        {
            "year": 2025,
            "month": 7,
            "price": 9000000,
            "url": "https://www.property24.com/property-values/99-grenache-close/val-de-vie-estate/paarl/western-cape/uq6vowa7e432zcjs3iokui5dyrrhmmcqoaq67y7ubmuuxdhbr2lazdypjvqcuvkxstciw2zssggmm",
            "streetNumber": "99",
            "streetName": "Grenache Close",
            "suburb": "Val de Vie Estate"
        },
        {
            "year": 2025,
            "month": 7,
            "price": 11600000,
            "url": "https://www.property24.com/property-values/835-muscat-crescent/val-de-vie-estate/paarl/western-cape/at7zmyaumi734afas5v3ygafhk6seljb23v2jstqjtgs6d4ucorr4oknxbymjfdq26prandavingg",
            "streetNumber": "835",
            "streetName": "Muscat Crescent",
            "suburb": "Val de Vie Estate"
        }
    ],
    "suburbTrendsUrl": "https://www.property24.com/paarl/val-de-vie-estate/property-trends/11726",
    "soldPriceSuburbUrl": "https://www.property24.com/property-values/val-de-vie-estate/paarl/western-cape/11726",
    "suburbName": "Val de Vie Estate",
    "cityName": "Paarl",
    "provinceName": "Western Cape",
    "suburbId": 11726,
    "cityId": 344,
    "provinceId": 9,
    "suburbResultsUrl": "https://www.property24.com/ResolveUrl/suburb?SuburbId=11726&ListingType=Sale",
    "status": "Active",
    "listingType": "Sale",
    "onShow": false,
    "repossessed": false,
    "auction": false,
    "listingNumber": "P24-116173830",
    "displayPrice": "R 140 000 000",
    "price": 140000000,
    "bedrooms": 12,
    "bathrooms": 12.5,
    "garages": 9,
    "parkingSpaces": 19,
    "size": {
        "unit": "m²",
        "value": 25635
    },
    "sizeType": "Erf",
    "thumbnailUrl": "https://images.prop24.com/360568809/Crop237x198",
    "tileType": "Normal",
    "midSizeImageUrl": "https://images.prop24.com/360568809/Crop600x400",
    "propertyTypeId": 4,
    "hasGroupedDuplicates": false,
    "agencyBrandingLogoType": "Default",
    "isPrivateListing": false,
    "isOffPortal": false,
    "banners": {},
    "badges": {
        "badges": []
    },
    "basicInfo": {
        "suburbName": "Val de Vie Estate",
        "cityName": "Paarl",
        "provinceName": "Western Cape",
        "suburbId": 11726,
        "cityId": 344,
        "provinceId": 9,
        "suburbResultsUrl": "https://www.property24.com/ResolveUrl/suburb?SuburbId=11726&ListingType=Sale",
        "status": "Active",
        "listingType": "Sale",
        "onShow": false,
        "repossessed": false,
        "auction": false,
        "listingNumber": "P24-116173830",
        "displayPrice": "R 140 000 000",
        "price": 140000000,
        "bedrooms": 12,
        "bathrooms": 12.5,
        "garages": 9,
        "parkingSpaces": 19,
        "size": {
            "unit": "m²",
            "value": 25635
        },
        "sizeType": "Erf",
        "thumbnailUrl": "https://images.prop24.com/360568809/Crop237x198",
        "tileType": "Boosted",
        "midSizeImageUrl": "https://images.prop24.com/360568809/Crop600x400",
        "propertyTypeId": 4,
        "hasGroupedDuplicates": false,
        "newFeaturedListingAgencyBranding": {
            "backgroundColourRGB": -11710635,
            "textColourRGB": -1,
            "thumbImageUrls": [
                "https://images.prop24.com/360568810/Crop162x108",
                "https://images.prop24.com/360568811/Crop162x108",
                "https://images.prop24.com/360568812/Crop162x108"
            ],
            "agentLogoUrl": "https://images.prop24.com/352244347/Crop204x306",
            "agentName": "Carryn Todd"
        },
        "agencyBrandingLogoUrl": "https://images.prop24.com/214388067/Ensure528x153",
        "agencyBrandingLogoType": "Wide",
        "isPrivateListing": false,
        "isOffPortal": false,
        "banners": {},
        "badges": {
            "badges": []
        },
        "promotedListingId": 2097976
    }
}

NameError: name 'false' is not defined